# World

> Run a policy in a pool of envs, to collect datasets or evaluate it.

In [ ]:
#| default_exp world.world

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import contextlib
from functools import partial
from pathlib import Path
from typing import Any, Callable, Iterator

import numpy as np

from stable_marl.envs.registry import make
from stable_marl.policy.base import Policy
from stable_marl.world.env_pool import EnvPool, episode_done
from stable_marl.data.format import Writer, get_format

In [ ]:
#| export
class World:
    """
    Runs a policy in ``num_envs`` copies of a registered environment, to collect datasets
    (:meth:`collect`) or evaluate the policy (:meth:`evaluate`)::

        world = World('MultiGrid-Empty-8x8-v0', num_envs=4, agents=2)
        world.set_policy(RandomPolicy(seed=0))
        world.collect('data/empty', episodes=100, seed=0)
        results = world.evaluate(episodes=50, seed=1000)

    Episode ``k`` of a run is reset with seed ``seed + k``, and the episodes are split between
    the envs in advance (env ``i`` plays episodes ``i, i + num_envs, ...``). The results thus
    only depend on `seed` (and the policy), not on which env happens to finish first, and short
    episodes are not over-represented.
    """
    def __init__(self, env_id: str, num_envs: int = 1, **env_kwargs):
        self.env_id, self.env_kwargs = env_id, env_kwargs
        self.envs = EnvPool([partial(make, env_id, **env_kwargs)] * num_envs)
        self.policy: Policy | None = None

    @property
    def num_envs(self) -> int:
        return self.envs.num_envs

    @property
    def num_agents(self) -> int:
        return self.envs.num_agents

    @property
    def metadata(self) -> dict[str, Any]:
        "What a dataset collected in this world records about where it comes from."
        return {'env_id': self.env_id, 'env_kwargs': self.env_kwargs, 'num_agents': self.num_agents}

    def set_policy(self, policy: Policy):
        "Attach the policy that :meth:`collect` and :meth:`evaluate` run."
        self.policy = policy
        policy.set_env(self.envs)

    def close(self):
        self.envs.close()

In [ ]:
#| export
from fastcore.utils import patch

@patch
def run(
    self: World,
    episodes: int,
    seed: int | None = None,
    options: dict | None = None,
    on_reset: Callable[[np.ndarray, dict], None] | None = None,
    on_step: Callable[..., None] | None = None) -> Iterator[tuple[int, int]]:
    """
    Play `episodes` episodes with the policy, yielding ``(env_index, episode_index)`` each time
    an episode ends (before that env is reset, so its final state can still be read).

    * ``on_reset(mask, obs)`` is called after the envs where `mask` is True started an episode
    * ``on_step(mask, actions, obs, rewards, terminations, truncations)`` after every step
      of the envs where `mask` is True
    """
    if self.policy is None:
        raise RuntimeError("No policy: call `set_policy` first")
    E = self.num_envs
    queues = [list(range(i, episodes, E)) for i in range(E)]   # episodes each env plays, in order
    current = np.array([q.pop(0) if q else -1 for q in queues])
    active = current >= 0
    seed_of = lambda k: None if seed is None or k < 0 else seed + int(k)

    obs, _ = self.envs.reset(seed=[seed_of(k) for k in current], options=options)
    self.policy.reset(active)
    if on_reset: on_reset(active, obs)

    while active.any():
        actions = self.policy.get_action(obs)
        obs, rewards, terms, truncs, _ = self.envs.step(actions, mask=active)
        if on_step: on_step(active, actions, obs, rewards, terms, truncs)

        done = active & episode_done(terms, truncs)
        for i in np.flatnonzero(done):
            yield int(i), int(current[i])
        if not done.any():
            continue

        restart = done & np.array([bool(q) for q in queues])
        current[done & ~restart], active[done & ~restart] = -1, False
        for i in np.flatnonzero(restart):
            current[i] = queues[i].pop(0)
        if restart.any():
            obs, _ = self.envs.reset(seed=[seed_of(k) for k in current], options=options, mask=restart)
            self.policy.reset(restart)
            if on_reset: on_reset(restart, obs)

In [ ]:
#| export
@patch
def evaluate(
    self: World,
    episodes: int,
    seed: int | None = None,
    options: dict | None = None) -> dict[str, Any]:
    """
    Play `episodes` episodes with the policy and return, indexed by episode:

    * ``episode_returns``: (episodes, num_agents) sum of each agent's rewards
    * ``episode_lengths``: (episodes,) number of steps
    * ``episode_successes``: (episodes,) whether the episode ended by termination
      (every agent terminated) rather than by the step limit
    * ``seeds``: (episodes,) reset seed of each episode (-1 if unseeded)

    and their summaries ``success_rate`` (in [0, 1]), ``mean_return`` (summed over agents)
    and ``mean_length``.
    """
    E, A = self.num_envs, self.num_agents
    results = {
        'episode_returns': np.zeros((episodes, A), np.float64),
        'episode_lengths': np.zeros(episodes, np.int64),
        'episode_successes': np.zeros(episodes, bool),
        'seeds': np.full(episodes, -1, np.int64),
    }
    returns, lengths, last_terms = np.zeros((E, A)), np.zeros(E, np.int64), np.zeros((E, A), bool)

    def on_reset(mask, obs):
        returns[mask], lengths[mask] = 0, 0

    def on_step(mask, actions, obs, rewards, terms, truncs):
        returns[mask] += rewards[mask]
        lengths[mask] += 1
        last_terms[mask] = terms[mask]

    for i, k in self.run(episodes, seed=seed, options=options, on_reset=on_reset, on_step=on_step):
        results['episode_returns'][k] = returns[i]
        results['episode_lengths'][k] = lengths[i]
        results['episode_successes'][k] = last_terms[i].all()
        results['seeds'][k] = self.envs.seeds[i]

    results['success_rate'] = float(results['episode_successes'].mean()) if episodes else 0.0
    results['mean_return'] = float(results['episode_returns'].sum(axis=1).mean()) if episodes else 0.0
    results['mean_length'] = float(results['episode_lengths'].mean()) if episodes else 0.0
    return results

In [ ]:
#| export
TRANSITION_KEYS = ('action', 'reward', 'terminated', 'truncated')


class _EpisodeRecorder:
    """
    Records one episode of one env as arrays, with time first:

    * ``obs.<key>`` (T + 1, num_agents, ...) and ``state`` (T + 1, ...): from the reset to the last step
    * ``action``, ``reward``, ``terminated``, ``truncated`` (T, num_agents): ``action[t]`` is taken
      in ``obs[t]`` and leads to ``obs[t + 1]``, with ``reward[t]``
    * ``episode.<key>`` (num_agents,): non-array observations (e.g. missions), from the reset
    """
    def __init__(self, obs_keys: list[str] | None):
        self.obs_keys = obs_keys
        self.steps: dict[str, list] = {}
        self.episode: dict[str, np.ndarray] = {}

    def add_obs(self, obs: dict[str, Any], i: int, state: np.ndarray):
        for key, value in obs.items():
            if self.obs_keys is not None and key not in self.obs_keys:
                continue
            if isinstance(value, np.ndarray):
                self.steps.setdefault(f'obs.{key}', []).append(value[i].copy())
            elif f'episode.{key}' not in self.episode:
                self.episode[f'episode.{key}'] = np.array([str(v) for v in value[i]])
        self.steps.setdefault('state', []).append(state)

    def add_transition(self, action, reward, terminated, truncated):
        for key, value in zip(TRANSITION_KEYS, (action, reward, terminated, truncated)):
            self.steps.setdefault(key, []).append(np.array(value))

    def finish(self) -> dict[str, np.ndarray]:
        episode = {key: np.stack(values) for key, values in self.steps.items()}
        episode['length'] = np.array(len(episode['state']) - 1)
        return {**episode, **self.episode}

In [ ]:
#| export
@patch
def collect(
    self: World,
    path: str | Path | None = None,
    episodes: int = 1,
    seed: int | None = None,
    options: dict | None = None,
    format: str = 'npz',
    writer: Writer | None = None,
    obs_keys: list[str] | None = None,
    mode: str = 'append',
    progress: bool = True):
    """
    Play `episodes` episodes with the policy and write them to `path`, in the dataset
    `format` (or to a ready-made `writer`).

    Each episode stores the observations (only `obs_keys` if given, e.g. to leave out the
    rendered ``'pov'``), the global states, the actions, rewards, terminations and
    truncations, its ``seed`` and ``episode_index`` (see :class:`_EpisodeRecorder`).
    `mode` is the writer mode: 'append', 'overwrite' or 'error' if the dataset exists.
    """
    if (path is None) == (writer is None):
        raise ValueError("Pass exactly one of `path` or `writer`")
    if writer is None:
        writer = get_format(format).open_writer(path, mode=mode, metadata=self.metadata)
    recorders: list[_EpisodeRecorder | None] = [None] * self.num_envs

    def on_reset(mask, obs):
        for i, state in zip(np.flatnonzero(mask), self.envs.state(mask)):
            recorders[i] = _EpisodeRecorder(obs_keys)
            recorders[i].add_obs(obs, i, state)

    def on_step(mask, actions, obs, rewards, terms, truncs):
        for i, state in zip(np.flatnonzero(mask), self.envs.state(mask)):
            recorders[i].add_transition(actions[i], rewards[i], terms[i], truncs[i])
            recorders[i].add_obs(obs, i, state)

    bar = _progress_bar(episodes, 'Collecting') if progress else contextlib.nullcontext()
    with writer as w, bar as pbar:
        def finished_episodes():
            for i, k in self.run(episodes, seed=seed, options=options, on_reset=on_reset, on_step=on_step):
                episode = recorders[i].finish()
                episode['episode_index'], episode['seed'] = np.array(k), np.array(self.envs.seeds[i])
                if pbar is not None: pbar.update(1)
                yield episode
        w.write_episodes(finished_episodes())


def _progress_bar(total: int, desc: str):
    try:
        from tqdm import tqdm
    except ImportError:
        return contextlib.nullcontext()
    return tqdm(total=total, desc=desc)

### Tests

In [ ]:
import tempfile
from stable_marl.policy import RandomPolicy
from stable_marl.envs.multigrid import GoToGoalPolicy
from stable_marl.data import load_episodes

world = World('MultiGrid-Empty-6x6-v0', num_envs=3, agents=2)
world.set_policy(RandomPolicy(seed=0))
res = world.evaluate(episodes=7, seed=100)
assert list(res['seeds']) == list(range(100, 107))
assert res['episode_returns'].shape == (7, 2) and 0 <= res['success_rate'] <= 1

# the expert reaches the goal, and results do not depend on the number of envs
for num_envs in (1, 3):
    world = World('MultiGrid-Empty-6x6-v0', num_envs=num_envs, agents=2)
    world.set_policy(GoToGoalPolicy())
    res = world.evaluate(episodes=5, seed=0)
    assert res['success_rate'] == 1.0, res
    if num_envs == 1: ref = res
    else: assert np.array_equal(ref['episode_lengths'], res['episode_lengths'])

# collect -> episodes on disk
with tempfile.TemporaryDirectory() as tmp:
    world = World('MultiGrid-Empty-6x6-v0', num_envs=2, agents=2)
    world.set_policy(RandomPolicy(seed=1))
    world.collect(tmp, episodes=3, seed=5, obs_keys=['image', 'direction', 'mission'], progress=False)
    episodes = load_episodes(tmp)
    assert episodes.num_episodes == 3 and episodes.metadata['env_id'] == 'MultiGrid-Empty-6x6-v0'
    ep = episodes.episode(0)
    T = int(ep['length'])
    assert ep['obs.image'].shape == (T + 1, 2, 7, 7, 3) and ep['state'].shape == (T + 1, 6, 6, 3)
    assert ep['action'].shape == ep['reward'].shape == ep['terminated'].shape == (T, 2)
    assert 'obs.pov' not in ep and ep['obs.mission'].shape == (T + 1, 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()